
# GCLF Reach Analysis

Repository: <a href="https://github.com/Pommers/gc_surveys_telescope">github.com/Pommers/gc_surveys_telescope</a> <br>

# GCLF Reach Analysis Notebook

## Purpose

This notebook performs an analysis on 8 contemporary observatory setups for GC survey capability. It will

1. Load the initial instrument parameters `csv`.
2. Generate GC modelling machinery
3. Add GC angular-size calculation
4. Normalize Band-passes for instruments
5. Add filter-specific columns
6. Save initial instrument analysis file


In [5]:

from pathlib import Path
import sys
import logging
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

%load_ext autoreload
%autoreload 2

def find_project_root(marker="src"):
    path = Path.cwd().resolve()
    for candidate in [path, *path.parents]:
        if (candidate / marker).exists():
            return candidate
    raise RuntimeError(
        f"Could not find project root containing '{marker}'"
    )

PROJECT_ROOT = find_project_root()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "data"
INTERMEDIATE_DIR = DATA_DIR / "intermediate"
# CATALOGUE_DIR = DATA_DIR / "catalogues" / "detection"
FIGURE_DIR = PROJECT_ROOT / "figures" 
CONFIG_DIR = PROJECT_ROOT / "config"

# CATALOGUE_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

from src.io_utils import setup_logging, load_config

setup_logging(verbose=True, name="Coma_GCs_survey")
logger = logging.getLogger("Coma_GCs_survey")

config = load_config(PROJECT_ROOT)

logger.info("Project root: %s", PROJECT_ROOT)
# logger.info("Catalogue directory: %s", CATALOGUE_DIR)
logger.info("Figure directory: %s", FIGURE_DIR)


Project root: /Users/richard/PycharmProjects/gc_surveys_telescope
Figure directory: /Users/richard/PycharmProjects/gc_surveys_telescope/figures


# Load the `inst.csv` file

In [6]:

inst = pd.read_csv(
    INTERMEDIATE_DIR / "instrument_parameters_v1.csv"
)

inst

,facility,instrument,regime,observing_mode,status,aperture_m,fov_width_arcsec,fov_height_arcsec,fov_area_deg2,pixel_scale_arcsec,...,D90_mpc,gc_rh_arcsec_100mpc,gc_resolution_ratio_100mpc,D_gc_rh_equals_psf_mpc,gc_filter_lambda_eff_um,gc_filter_lambda_min_um,gc_filter_lambda_max_um,gc_filter_basis,parameter_version,filter_values_status
0,HST,ACS/WFC,targeted_space,directed,operational,2.4,202.0,202.0,0.003148,0.049,...,NaN,0.006188,0.068757,6.875688,0.8060,0.730,0.880,published pivot + approximate broadband limits,v1_firstpass,first_pass
1,JWST,NIRCam SW,targeted_space,directed,operational,6.5,NaN,NaN,0.002694,0.031,...,NaN,0.006188,0.123762,12.376238,1.5010,1.331,1.668,STScI commissioning total-system throughput,v1_firstpass,first_pass
2,Subaru,HSC,wide_ground,survey+directed,operational,8.2,5400.0,5400.0,1.767146,0.168,...,NaN,0.006188,0.010314,1.031353,0.7700,0.700,0.850,HSC published filter curve; first-pass limits,v1_firstpass,first_pass
3,Rubin,LSSTCam,wide_ground,survey,operational,8.4,NaN,NaN,9.600000,0.200,...,NaN,0.006188,0.008840,0.884017,0.7500,0.690,0.820,LSST total-system band approximation,v1_firstpass,first_pass
4,Euclid,VIS,wide_space,survey,operational,1.2,NaN,NaN,0.540000,0.100,...,NaN,0.006188,0.034378,3.437844,0.7200,0.550,0.900,Euclid VIS nominal broad passband,v1_firstpass,first_pass
5,Roman,WFI,wide_space,survey+directed,forthcoming,2.4,NaN,NaN,0.281000,0.110,...,NaN,0.006188,NaN,NaN,0.8696,0.795,0.944,Roman WFI pivot and FWHM,v1_firstpass,first_pass
6,ELT,MICADO + MORFEO,elt_ao,directed,planned,39.0,50.5,50.5,0.000197,0.004,...,NaN,0.006188,NaN,NaN,1.2500,1.150,1.350,standard J-band approximation,v1_firstpass,first_pass
7,TMT,IRIS + NFIRAOS,elt_ao,directed,planned,30.0,34.0,34.0,0.000089,0.004,...,NaN,0.006188,NaN,NaN,1.2500,1.150,1.350,standard J-band approximation,v1_firstpass,first_pass


# Calculate $M_\mathrm{GCLF,TO}$

## Determine $I$-band Corrections

For $M_\mathrm{GCLF,TO}$, we initially start from the empirical $I$-band anchor at

$$M^\mathrm{TO}_I\approx−8.46$$

and transform away from $I$ using a single old-GC population assumption, rather than mixing unrelated turnover measurements from different galaxies. Kundu & Whitmore give the optical anchor, while empirical optical–NIR GC work shows that old clusters are well reproduced by SSP models in this regime [(e.g., Bruzual & Charlot., 2003](https://ui.adsabs.harvard.edu/abs/2003MNRAS.344.1000B/abstract); [Pessev et al., 2008)](https://ui.adsabs.harvard.edu/abs/2008MNRAS.385.1535P/abstract)

As we are looking at different regimes (e.g., directed, survey) we should add this to the GCLF determination ...

In [7]:
inst["M_gclf_to"] = np.nan
inst["M_gclf_to_basis"] = ""


and treat the facilities in three groups


In [8]:
# First-pass grouping only

optical_I_like = ["HST", "Subaru", "Rubin", "Euclid", "Roman"]
near_ir = ["ELT", "TMT"]
deep_nir = ["JWST"]


For the optical group, we can reasonably start very close to the empirical $I$-band turnover and then refine the small filter offsets later.

For $J$ and $\mathrm{F150W}$, we should not guess a correction blindly. There is empirical optical–NIR work on old GCs, including [Pessev et al. (2008)](https://ui.adsabs.harvard.edu/abs/2008MNRAS.385.1535P/abstract) and [Blakeslee et al. (2012)](https://ui.adsabs.harvard.edu/abs/2012ApJ...746...88B/abstract); importantly, Pessev et al. find that standard SSP models reproduce the integrated colors of $>10$ Gyr clusters reasonably well. That gives us a defensible route to an $I-J$ and $I-\mathrm{F150W}$ correction.

So, in practical terms, we create a dataframe with all the info needed for the gclf model

In [9]:
gclf_band_model = pd.DataFrame({
    "facility": inst["facility"],
    "filter": inst["gc_filter"],
    "lambda_eff_um": inst["gc_filter_lambda_eff_um"],
    "M_to_ab": np.nan,
    "delta_from_I": np.nan,
    "basis": ""
})


then populate the optical filters first - **Note - ONLY for exploratory plot!** - Euclid $\mathrm{VIS}$ and Roman $\mathrm{F087}$ are approximated as $I$-like.

In [10]:
# define GCLF TO in I-filter band for optical
M_TO_I = -8.46

approx_optical = {
    "HST":    0.00,
    "Subaru": 0.00,
    "Rubin":  0.00,
    "Euclid": 0.00,
    "Roman":  0.00,
}

for facility, delta in approx_optical.items():
    mask = gclf_band_model["facility"] == facility

    gclf_band_model.loc[mask, "delta_from_I"] = delta
    gclf_band_model.loc[mask, "M_to_ab"] = M_TO_I + delta
    gclf_band_model.loc[mask, "basis"] = "first-pass I-like approximation"

gclf_band_model

,facility,filter,lambda_eff_um,M_to_ab,delta_from_I,basis
0,HST,F814W,0.8060,-8.46,0.0,first-pass I-like approximation
1,JWST,F150W,1.5010,NaN,NaN,
2,Subaru,i2,0.7700,-8.46,0.0,first-pass I-like approximation
3,Rubin,i,0.7500,-8.46,0.0,first-pass I-like approximation
4,Euclid,VIS,0.7200,-8.46,0.0,first-pass I-like approximation
5,Roman,F087,0.8696,-8.46,0.0,first-pass I-like approximation
6,ELT,J,1.2500,NaN,NaN,
7,TMT,J,1.2500,NaN,NaN,


Pessev et al. is useful as an empirical validation paper: it gives integrated 2MASS $JHK_s$ photometry for Magellanic Cloud clusters and composite colors such as $V−J$, $V−K_s$, and $J−K_s$, and shows that several SSP models reproduce the colors of old $>10$ Gyr clusters reasonably well. But it does not give us the clean $I−J$ or $\mathrm{F814W}–\mathrm{F150W}$ transformation we actually want, and its photometry is in the 2MASS/Vega system. So we'd immediately be doing extra conversions and mixing systems.

There is newer empirical work — including actual JWST observations of old GCs. [Nardiello et al. (2022)](https://ui.adsabs.harvard.edu/abs/2022MNRAS.517..484N/abstract) obtained F090W/F150W photometry of M92, and [Milone et al. (2023)](https://ui.adsabs.harvard.edu/abs/2023MNRAS.522.2429M/abstract) modeled old GC stars directly in NIRCam filters. Even more directly relevant to our extragalactic problem, [Berkheimer et al. (2024)](https://ui.adsabs.harvard.edu/abs/2024ApJ...964L..29B/abstract) used $\mathrm{F090W}/\mathrm{F150W}$ GC photometry around VV 191a; their models imply a GC luminosity-function peak around $M_\mathrm{AB}\sim−9.4\pm0.2$ in $\mathrm{F090W}$, although their data do not quite reach the turnover. And work from our friend William [Harris et al. (2025)](https://ui.adsabs.harvard.edu/abs/2025ApJ...993..210H/abstract) has explicit evaluated NIRCam filters for GC systems over redshift.


So, a cleaner modern methodology for this paper would be

>Anchor the GCLF turnover mass/luminosity empirically in F814W, then use one modern SSP SED to calculate synthetic AB colors in all eight adopted passbands.

In other words, rather than saying

$$M_J^\mathrm{TO}=M_I^\mathrm{TO}−(I−J)_\mathrm{Pessev},$$

we calculate

$$\Delta_X=m_X^\mathrm{SSP}−m_\mathrm{F814W}^\mathrm{SSP}$$

for a representative old GC, and then

$$M_X^\mathrm{TO}=−8.46+\Delta_X.$$

That automatically handles $\mathrm{F150W}, \mathrm{F087}$, Euclid VIS, Rubin $i$, etc., rather than trying to find empirical transformations for every modern filter.

## What SSP?

For a quick first pass, FSPS + MIST looks very attractive. Modern published work routinely uses FSPS with MIST isochrones and synthetic photometry in both HST/ACS and JWST/NIRCam filters; [e.g., Wan et al. (2024)](https://ui.adsabs.harvard.edu/abs/2024MNRAS.532.4002W/abstract) explicitly generate $\mathrm{F814W}$ and $\mathrm{F150W}$ photometry this way.

So the way forward is to choose a simple representative population like

- age_gyr = 12.0
- feh = -1.0

but later we can repeat with variations

- age_gyr = [10, 12, 13]
- feh = [-1.5, -1.0, -0.5]

and the exact mass normalization cancels in the color transformation. We don't need to know the SSP's absolute mass yet; we only need its relative flux between F814W and the target passband.

Pessev et al. becomes a nice supporting citation saying old-cluster optical–NIR colors are empirically reasonably reproduced by SSPs, rather than being responsible for the numerical conversion.



## Set up the minimal FSPS/MIST calculation

Attempt to get F814W → J/F150W offsets

FSPS can give us a 12-Gyr SSP spectrum directly in $L_\odot/\mathrm{Hz}$, with MIST available in the standard setup, and we can integrate that spectrum through our first-pass top-hat bandpasses using the min/max wavelengths we already stored. That keeps everything internally consistent and gets us to $M_\mathrm{TO} today. FSPS documents `StellarPopulation`, `get_spectrum()`, and AB-magnitude generation directly.

In [14]:
# import fsps

# sp = fsps.StellarPopulation()


In [ ]:
# sp = fsps.StellarPopulation(
#     compute_vega_mags=False,   # AB system
#     zcontinuous=1,
#     sfh=0,                     # SSP
#     logzsol=-1.0,              # first-pass old GC metallicity
#     add_neb_emission=False,
#     add_dust_emission=False,
# )

# print(sp.libraries)

buuuuutt...

this requires a full SPS_HOME/model-data setup, so in the first instance I'll go with some defensible values to do some first pass work.

# First pass - defensible values!

For a first-pass we can use two empirical NIR anchors rather than fight the SPS software...!

- J-band: [Wang et al. (2014)](https://ui.adsabs.harvard.edu/abs/2014arXiv1404.4444W/abstract) measure an M31 GC GCLF peak at $J_0=15.348_{−0.208}^{+0.206}$ in 2MASS $J$. Using the 2MASS $J$ zero-point of 1594 Jy gives $J_\mathrm{AB}−J_\mathrm{Vega}≃0.91$ mag. With the standard M31 distance modulus $\mu\simeq24.47$, this corresponds to roughly

$$M_{J,\mathrm{AB}}^\mathrm{TO}\simeq−8.21.$$

- $\mathrm{F150W}$: recent JWST GC work [(e.g., Keatley & Harris, 2025)](https://ui.adsabs.harvard.edu/abs/2025ApJ...990...67K/abstract) quotes an expected turnover around $M_\mathrm{F150W}\simeq−8.3\ \mathrm{AB}$.

This is pleasantly consistent, because the NIR turnover in $\mathrm{AB}$ magnitudes is not dramatically brighter than $\mathrm{F814W}$. The huge Vega-system $I−J$ colors mostly disappear once both bands are expressed in $\mathrm{AB}$.

In [15]:
M_TO_F814W = -8.46

firstpass_Mto = {
    "HST":    -8.46,  # F814W empirical anchor
    "JWST":   -8.30,  # F150W, literature first-pass
    "Subaru": -8.46,  # i2 ~ I-like approximation
    "Rubin":  -8.46,  # i ~ I-like approximation
    "Euclid": -8.46,  # broad VIS; provisional
    "Roman":  -8.46,  # F087 ~ I/z-like; provisional
    "ELT":    -8.21,  # J, empirical M31-based estimate
    "TMT":    -8.21,  # same J-band assumption
}

# flag the basis of the above values
firstpass_basis = {
    "HST":    "empirical F814W GCLF anchor",
    "JWST":   "literature F150W turnover approximation",
    "Subaru": "F814W-equivalent first-pass approximation",
    "Rubin":  "F814W-equivalent first-pass approximation",
    "Euclid": "F814W-equivalent first-pass approximation",
    "Roman":  "F814W-equivalent first-pass approximation",
    "ELT":    "2MASS J GCLF; M31 empirical estimate converted to AB",
    "TMT":    "2MASS J GCLF; M31 empirical estimate converted to AB",
}


Update the table...

In [18]:
for facility, Mto in firstpass_Mto.items():
    mask = gclf_band_model["facility"] == facility

    gclf_band_model.loc[mask, "M_to_ab"] = Mto
    gclf_band_model.loc[mask, "delta_from_F814W"] = (
        Mto - M_TO_F814W
    )
    gclf_band_model.loc[mask, "basis"] = firstpass_basis[facility]

gclf_band_model["M_to_uncertainty_mag"] = 0.20

gclf_band_model

,facility,filter,lambda_eff_um,M_to_ab,delta_from_I,basis,delta_from_F814W,M_to_uncertainty_mag
0,HST,F814W,0.8060,-8.46,0.0,empirical F814W GCLF anchor,0.00,0.2
1,JWST,F150W,1.5010,-8.30,NaN,literature F150W turnover approximation,0.16,0.2
2,Subaru,i2,0.7700,-8.46,0.0,F814W-equivalent first-pass approximation,0.00,0.2
3,Rubin,i,0.7500,-8.46,0.0,F814W-equivalent first-pass approximation,0.00,0.2
4,Euclid,VIS,0.7200,-8.46,0.0,F814W-equivalent first-pass approximation,0.00,0.2
5,Roman,F087,0.8696,-8.46,0.0,F814W-equivalent first-pass approximation,0.00,0.2
6,ELT,J,1.2500,-8.21,NaN,2MASS J GCLF; M31 empirical estimate converted...,0.25,0.2
7,TMT,J,1.2500,-8.21,NaN,2MASS J GCLF; M31 empirical estimate converted...,0.25,0.2


In [20]:
gclf_band_model = gclf_band_model.drop(
    columns=["delta_from_I"],
    errors="ignore"
)

gclf_band_model.loc[
    gclf_band_model["facility"] == "HST",
    "M_to_uncertainty_mag"
] = 0.10

gclf_band_model

,facility,filter,lambda_eff_um,M_to_ab,basis,delta_from_F814W,M_to_uncertainty_mag
0,HST,F814W,0.8060,-8.46,empirical F814W GCLF anchor,0.00,0.1
1,JWST,F150W,1.5010,-8.30,literature F150W turnover approximation,0.16,0.2
2,Subaru,i2,0.7700,-8.46,F814W-equivalent first-pass approximation,0.00,0.2
3,Rubin,i,0.7500,-8.46,F814W-equivalent first-pass approximation,0.00,0.2
4,Euclid,VIS,0.7200,-8.46,F814W-equivalent first-pass approximation,0.00,0.2
5,Roman,F087,0.8696,-8.46,F814W-equivalent first-pass approximation,0.00,0.2
6,ELT,J,1.2500,-8.21,2MASS J GCLF; M31 empirical estimate converted...,0.25,0.2
7,TMT,J,1.2500,-8.21,2MASS J GCLF; M31 empirical estimate converted...,0.25,0.2
